In [1]:
# ============================================================
# CELL 1: Setup & Verify Pipeline Artifacts
# ============================================================
import json
import os
import joblib
import numpy as np
import pandas as pd

print("="*60)
print("STAGE 3 PREPARATION - ARTIFACT EXPORT")
print("="*60)

# Paths
PREPROCESS_DIR = "../outputs/preprocess"
PROCESSED_DIR = "../outputs/processed"
MODELS_DIR = "../outputs/models"
ART_DIR = "../stage3_zk/artifacts"
TEST_VEC_DIR = "../stage3_zk/test_vectors"

os.makedirs(ART_DIR, exist_ok=True)
os.makedirs(TEST_VEC_DIR, exist_ok=True)

# Verify all required files exist
required_files = {
    "feature_names": os.path.join(PREPROCESS_DIR, "feature_names.json"),
    "X_test": os.path.join(PROCESSED_DIR, "X_test.npy"),
    "y_test": os.path.join(PROCESSED_DIR, "y_test.npy"),
    "logreg_model": os.path.join(MODELS_DIR, "logreg_baseline.pkl"),
}

for name, path in required_files.items():
    assert os.path.exists(path), f"Missing: {name} at {path}"
    print(f"✅ Found: {name}")

print("\n" + "="*60)

STAGE 3 PREPARATION - ARTIFACT EXPORT
✅ Found: feature_names
✅ Found: X_test
✅ Found: y_test
✅ Found: logreg_model



In [ ]:
# ============================================================
# CELL 2: Load & Freeze Feature Order (n = auto-detected)
# ============================================================
with open(required_files["feature_names"], "r", encoding="utf-8") as f:
    FEATURE_NAMES = json.load(f)

n_features = len(FEATURE_NAMES)

print(f"Number of features (n): {n_features}")
print(f"First 10 features: {FEATURE_NAMES[:10]}")
print(f"Last 5 features: {FEATURE_NAMES[-5:]}")

# Export to Stage 3 artifacts
with open(os.path.join(ART_DIR, "feature_order.json"), "w", encoding="utf-8") as f:
    json.dump(FEATURE_NAMES, f, indent=2)

print(f"\n✅ Saved: {os.path.join(ART_DIR, 'feature_order.json')}")

Number of features (n): 104
First 10 features: ['duration', 'src_bytes', 'dst_bytes', 'src_pkts', 'dst_pkts', 'src_ip_bytes', 'dst_ip_bytes', 'missed_bytes', 'src_port', 'dst_port']
Last 5 features: ['dns_AA', 'dns_RD', 'dns_RA', 'ssl_resumed', 'ssl_established']

✅ Saved: ../stage3_zk/artifacts\feature_order.json


In [3]:
# ============================================================
# CELL 3: Create Semantic Group Mapping (5 groups)
# ============================================================
GROUP_IDS = {
    "Protocol": 1,
    "Application": 2,
    "ConnectionState": 3,
    "Ports": 4,
    "TrafficVolume": 5,
}

def feature_to_group_id(fname: str) -> int:
    """Map feature name to semantic group ID (1-5)."""
    # Protocol (one-hot)
    if fname.startswith("proto_"):
        return GROUP_IDS["Protocol"]
    
    # Ports (numeric)
    if fname in {"src_port", "dst_port"}:
        return GROUP_IDS["Ports"]
    
    # ConnectionState (one-hot)
    if fname.startswith("conn_state_"):
        return GROUP_IDS["ConnectionState"]
    
    # TrafficVolume (numeric features)
    if (any(key in fname for key in ["_bytes", "_pkts"]) 
        or fname in {"duration", "missed_bytes", "http_request_body_len", "http_response_body_len"}):
        return GROUP_IDS["TrafficVolume"]
    
    # Application (service/http/ssl/dns/weird + booleans)
    if (fname.startswith("service_") or fname.startswith("http_") 
        or fname.startswith("ssl_") or fname.startswith("dns_") 
        or fname.startswith("weird_")):
        return GROUP_IDS["Application"]
    
    raise ValueError(f"Feature '{fname}' does not map to any semantic group!")

# Generate group mapping
group_ids = [feature_to_group_id(nm) for nm in FEATURE_NAMES]

# Sanity check: no unmapped features
assert all(g in {1,2,3,4,5} for g in group_ids), "Found unmapped feature!"

# Count by group
group_counts = {g: group_ids.count(g) for g in range(1, 6)}
print("\nGroup distribution:")
for gid, count in sorted(group_counts.items()):
    gname = [k for k, v in GROUP_IDS.items() if v == gid][0]
    print(f"  Group {gid} ({gname}): {count} features")

# Export group map
group_map = {
    "n_features": n_features,
    "n_groups": 5,
    "groups": list(GROUP_IDS.keys()),
    "group_to_id": GROUP_IDS,
    "feature_index_to_group_id": group_ids,
    "feature_index_to_group_name": [
        [k for k, v in GROUP_IDS.items() if v == feature_to_group_id(nm)][0] 
        for nm in FEATURE_NAMES
    ]
}

with open(os.path.join(ART_DIR, "group_map.json"), "w", encoding="utf-8") as f:
    json.dump(group_map, f, indent=2)

print(f"\n✅ Saved: {os.path.join(ART_DIR, 'group_map.json')}")


Group distribution:
  Group 1 (Protocol): 3 features
  Group 2 (Application): 76 features
  Group 3 (ConnectionState): 13 features
  Group 4 (Ports): 2 features
  Group 5 (TrafficVolume): 10 features

✅ Saved: ../stage3_zk/artifacts\group_map.json


In [4]:
# ============================================================
# CELL 4: Extract Model Weights (float → int quantization)
# ============================================================
logreg_model = joblib.load(required_files["logreg_model"])

# Extract float weights and bias
w_float = logreg_model.coef_.reshape(-1).astype(float)
b_float = float(logreg_model.intercept_[0])

# Sanity check dimensions
assert len(w_float) == n_features, \
    f"Weight dimension {len(w_float)} != n_features {n_features}"

print(f"Loaded Logistic Regression model:")
print(f"  Number of weights: {len(w_float)}")
print(f"  Bias (float): {b_float:.6f}")
print(f"  Weight range: [{w_float.min():.6f}, {w_float.max():.6f}]")

# Quantization scales (FIXED)
Sx = 2**16  # Input scale: 65536
Sw = 2**12  # Weight scale: 4096

# Quantize to integers
w_int = np.round(w_float * Sw).astype(np.int64)
b_int = int(np.round(b_float * Sx * Sw))

print(f"\nQuantization:")
print(f"  Sx (input scale): {Sx}")
print(f"  Sw (weight scale): {Sw}")
print(f"  |w_int| max: {int(np.max(np.abs(w_int)))}")
print(f"  b_int: {b_int}")

# Export model_public.json
model_public = {
    "n": n_features,
    "Sx": Sx,
    "Sw": Sw,
    "w_int": w_int.tolist(),
    "b_int": b_int,
    "scaling_notes": "score_int = sum(x_int[i] * w_int[i]) + b_int; y_hat = (score_int >= 0)"
}

with open(os.path.join(ART_DIR, "model_public.json"), "w", encoding="utf-8") as f:
    json.dump(model_public, f, indent=2)

print(f"\n✅ Saved: {os.path.join(ART_DIR, 'model_public.json')}")

Loaded Logistic Regression model:
  Number of weights: 104
  Bias (float): -1.976431
  Weight range: [-29.816876, 10.112389]

Quantization:
  Sx (input scale): 65536
  Sw (weight scale): 4096
  |w_int| max: 122130
  b_int: -530544174

✅ Saved: ../stage3_zk/artifacts\model_public.json


In [5]:
# ============================================================
# CELL 5: Estimate Bounds (for ZK field constraints)
# ============================================================
# Load test data (memmap for efficiency)
X_test = np.load(required_files["X_test"], mmap_mode="r")
y_test = np.load(required_files["y_test"], mmap_mode="r")

print(f"Test set shape: {X_test.shape}")
print(f"Test set dtype: {X_test.dtype}")

# Quantize test samples
X_test_int = np.round(X_test * Sx).astype(np.int64)

# Compute bounds on quantized inputs
max_abs_x_int = int(np.max(np.abs(X_test_int)))
print(f"\nmax(|x_int|) across all test samples: {max_abs_x_int}")

# Estimate score bounds (use subsample for speed)
rng = np.random.default_rng(42)
n_samples = min(5000, len(X_test_int))
idx_sample = rng.choice(len(X_test_int), size=n_samples, replace=False)

scores_int = X_test_int[idx_sample] @ w_int + b_int
max_abs_score = int(np.max(np.abs(scores_int)))

print(f"\nScore bounds (from {n_samples} samples):")
print(f"  max(|score_int|): {max_abs_score}")
print(f"  score_int range: [{scores_int.min()}, {scores_int.max()}]")

# Export bounds
bounds = {
    "max_abs_x_int": max_abs_x_int,
    "max_abs_w_int": int(np.max(np.abs(w_int))),
    "max_abs_score_int": max_abs_score,
    "note": "These bounds must fit in ZK field (typically ~254 bits for BN254 curve)"
}

with open(os.path.join(ART_DIR, "bounds.json"), "w", encoding="utf-8") as f:
    json.dump(bounds, f, indent=2)

print(f"\n✅ Saved: {os.path.join(ART_DIR, 'bounds.json')}")

Test set shape: (502628, 104)
Test set dtype: float32

max(|x_int|) across all test samples: 297270816

Score bounds (from 5000 samples):
  max(|score_int|): 22988183559
  score_int range: [-22988183559, 8230058204]

✅ Saved: ../stage3_zk/artifacts\bounds.json


In [6]:
# ============================================================
# CELL 6: Generate Test Vectors (for circuit testing)
# ============================================================
# Select 3 diverse samples: 1 TP, 1 TN, 1 FN
y_pred_test = logreg_model.predict(X_test)

tp_idx = np.where((y_test == 1) & (y_pred_test == 1))[0]
tn_idx = np.where((y_test == 0) & (y_pred_test == 0))[0]
fn_idx = np.where((y_test == 1) & (y_pred_test == 0))[0]

test_cases = [
    ("TP_attack", tp_idx[0] if len(tp_idx) > 0 else 0),
    ("TN_normal", tn_idx[0] if len(tn_idx) > 0 else 1),
    ("FN_attack", fn_idx[0] if len(fn_idx) > 0 else 2),
]

for i, (label, idx) in enumerate(test_cases, start=1):
    x_float = X_test[idx]
    x_int = X_test_int[idx]
    y_true = int(y_test[idx])
    y_pred = int(y_pred_test[idx])
    
    # Compute score
    score_int = int(x_int @ w_int + b_int)
    
    # Compute group contributions
    c_int = x_int * w_int
    abs_c_int = np.abs(c_int)
    
    G = np.zeros(6, dtype=np.int64)  # G[0] unused, G[1..5] for groups
    for feat_idx, gid in enumerate(group_ids):
        G[gid] += abs_c_int[feat_idx]
    
    # Find top-3 groups
    group_contribs = [(gid, int(G[gid])) for gid in range(1, 6)]
    group_contribs.sort(key=lambda x: x[1], reverse=True)
    top3_groups = [g[0] for g in group_contribs[:3]]
    
    test_vector = {
        "sample_id": int(idx),
        "label": label,
        "y_true": y_true,
        "y_pred": y_pred,
        "x_int": x_int.tolist(),
        "score_int": score_int,
        "y_hat": int(score_int >= 0),
        "group_contributions": {str(gid): int(G[gid]) for gid in range(1, 6)},
        "top3_groups": top3_groups,
        "feature_order_matches": "feature_order.json"
    }
    
    out_path = os.path.join(TEST_VEC_DIR, f"test_sample_{i}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(test_vector, f, indent=2)
    
    print(f"\n✅ Saved test vector {i}: {label} (idx={idx})")
    print(f"   y_true={y_true}, y_pred={y_pred}, score={score_int}, top3={top3_groups}")

print(f"\n{'='*60}")
print("STAGE 3 ARTIFACTS COMPLETE!")
print(f"{'='*60}")


✅ Saved test vector 1: TP_attack (idx=0)
   y_true=1, y_pred=1, score=390139428, top3=[2, 1, 5]

✅ Saved test vector 2: TN_normal (idx=30)
   y_true=0, y_pred=0, score=-661754717, top3=[2, 3, 1]

✅ Saved test vector 3: FN_attack (idx=16)
   y_true=1, y_pred=0, score=-307632372, top3=[2, 1, 5]

STAGE 3 ARTIFACTS COMPLETE!
